# Instalasi dan Impor Library

In [ ]:
!pip install nltk Sastrawi pandas numpy tabulate

import nltk
import math
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tabulate import tabulate

In [ ]:
# Mengunduh resource NLTK yang diperlukan
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')
print("Instalasi dan impor library selesai.")

# Impor Data

In [ ]:
from google.colab import files
import sys

# A. Upload File MESIN (Dapur)
print("upload file versi mesin (sudah preprocessing)")
uploaded_mesin = files.upload()
filename_mesin = list(uploaded_mesin.keys())[0]

# baca file mesin → tiap baris = 1 kalimat
with open(filename_mesin, "r", encoding="utf-8") as f:
    processed_sentences = f.read().splitlines()

print(f">>> File Mesin '{filename_mesin}' dibaca: {len(processed_sentences)} baris.")

# B. Upload File MANUSIA (Etalase)
print("\nupload file versi manusia (asli)")
uploaded_manusia = files.upload()
filename_manusia = list(uploaded_manusia.keys())[0]

# baca file manusia → juga per baris
with open(filename_manusia, "r", encoding="utf-8") as f:
    # Kita pakai .splitlines() juga
    sentences = f.read().splitlines()

print(f">>> File Manusia '{filename_manusia}' dibaca: {len(sentences)} baris.")

# C. Validasi Paralel (biar gak error pas diproses nanti)
if len(sentences) != len(processed_sentences):
    print(f"\nJumlah baris tidak sama, ada yang tidak sinkron.")
    print(f"Ver Asli  : {len(sentences)} baris.")
    print(f"Ver Mesin : {len(processed_sentences)} baris.")
    print("Jumlah baris harus sama persis. tolong dicek lagi.")
    sys.exit("Proses dihentikan karena data tidak paralel.")
else:
    print("\n>>> Validasi sukses dan siap diproses.")

# LexRank dengan IDF-Modified Cosine Similarity

1. INPUT KALIMAT (HASIL PREPROCESSING)

Digunakan untuk mengubah kalimat hasil preprocessing menjadi token kata dan menghitung jumlah total kalimat (N) yang digunakan dalam perhitungan IDF dan PageRank pada algoritma LexRank.



In [ ]:
# Pastikan setiap kalimat sudah berupa list token
processed_tokens = [s.split() for s in processed_sentences]

N = len(processed_tokens)
print(f"\nJumlah Kalimat (N): {N}")

2. TERM FREQUENCY (TF)

Digunakan untuk menghitung Term Frequency (TF) setiap kata pada masing-masing kalimat. Nilai TF tersebut kemudian digunakan dalam perhitungan IDF-Modified Cosine Similarity pada algoritma LexRank.

In [ ]:
tf = [Counter(tokens) for tokens in processed_tokens]

# TABEL TF
tf_rows = []
for i, sent_tf in enumerate(tf):
    row = {"Kalimat": f"K{i+1}"}
    row.update(sent_tf)
    tf_rows.append(row)

tf_df = pd.DataFrame(tf_rows).fillna(0)

print("\n=== TABEL TERM FREQUENCY (TF) ===")
print(tabulate(tf_df, headers="keys", tablefmt="pipe", showindex=False))

3. DOCUMENT FREQUENCY (DF)

  Digunakan untuk menghitung Document Frequency (DF), yaitu jumlah kalimat yang mengandung suatu kata. Nilai DF ini kemudian digunakan dalam perhitungan IDF pada tahap pembentukan bobot kemiripan antar kalimat dalam algoritma LexRank.

In [ ]:
df = defaultdict(int)
for sent_tf in tf:
    for word in sent_tf:
        df[word] += 1

df_df = pd.DataFrame({
    "Term": df.keys(),
    "DF": df.values()
})

print("\n=== TABEL DOCUMENT FREQUENCY (DF) ===")
print(tabulate(df_df, headers="keys", tablefmt="pipe", showindex=False))

4. INVERSE DOCUMENT FREQUENCY (IDF)

Bagian ini menghitung nilai IDF berdasarkan jumlah kalimat yang memuat suatu kata. Nilai ini digunakan sebagai bobot dalam perhitungan IDF-Modified Cosine Similarity pada algoritma LexRank.

In [ ]:
idf = {word: math.log(N / freq) for word, freq in df.items()}

idf_df = pd.DataFrame({
    "Term": idf.keys(),
    "DF": [df[w] for w in idf.keys()],
    "N": N,
    "IDF": [round(idf[w], 4) for w in idf.keys()]
})

print("\n=== TABEL IDF ===")
print(tabulate(idf_df, headers="keys", tablefmt="pipe", showindex=False))

5. TF × IDF (VEKTOR KALIMAT)

Digunakan agar kalimat bisa dibandingkan secara numerik sehingga LexRank dapat menentukan kalimat mana yang paling sentral dalam dokumen.

In [ ]:
tfidf_vectors = []

for sent_tf in tf:
    vec = {}
    for word in sent_tf:
        vec[word] = sent_tf[word] * idf[word]
    tfidf_vectors.append(vec)

tfidf_df = pd.DataFrame(tfidf_vectors).fillna(0)
tfidf_df.index = [f"K{i+1}" for i in range(N)]

print("\n=== TABEL VEKTOR TF × IDF ===")
print(tabulate(tfidf_df, headers="keys", tablefmt="pipe", floatfmt=".4f"))

In [ ]:
# Hitung document frequency (jumlah kalimat dengan nilai > 0 per kata)
df_count = (tfidf_df > 0).sum(axis=0)

# Urutkan dari yang paling sering muncul
df_sorted = df_count.sort_values(ascending=False)

print("\n=== 15 Kata Paling Sering Muncul (Document Frequency Tertinggi) ===")
print(df_sorted.head(15))

6. IDF-MODIFIED COSINE SIMILARITY

Fungsi ini digunakan untuk menghitung bobot keterkaitan antarkalimat menggunakan idf-modified cosine similarity sebagaimana didefinisikan dalam LexRank oleh Erkan dan Radev (2004).

In [ ]:
def idf_modified_cosine(tf_i, tf_j, idf):
    num = sum(tf_i[w] * tf_j[w] * idf[w]**2
              for w in tf_i if w in tf_j)

    den_i = sum((tf_i[w] * idf[w])**2 for w in tf_i)
    den_j = sum((tf_j[w] * idf[w])**2 for w in tf_j)

    if den_i == 0 or den_j == 0:
        return 0.0

    return num / (math.sqrt(den_i) * math.sqrt(den_j))

7. ADJACENCY MATRIX (CONTINUOUS GRAPH)

Bagian ini digunakan untuk membangun adjacency matrix graf LexRank, di mana setiap simpul merepresentasikan satu kalimat dan bobot sisi antar simpul dihitung menggunakan idf-modified cosine similarity sesuai dengan pendekatan continuous LexRank.

In [ ]:
sim_matrix = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        if i != j:
            sim_matrix[i][j] = idf_modified_cosine(tf[i], tf[j], idf)

sim_df = pd.DataFrame(
    sim_matrix,
    index=[f"K{i+1}" for i in range(N)],
    columns=[f"K{i+1}" for i in range(N)]
)

print("\n=== ADJACENCY MATRIX (IDF-MODIFIED COSINE) ===")
print(tabulate(sim_df, headers="keys", tablefmt="pipe", floatfmt=".4f"))

8. NORMALISASI BARIS (TRANSITION MATRIX)

In [ ]:
row_normalized = sim_matrix.copy()

for i in range(N):
    s = row_normalized[i].sum()
    if s != 0:
        row_normalized[i] /= s

trans_df = pd.DataFrame(
    row_normalized,
    index=[f"K{i+1}" for i in range(N)],
    columns=[f"K{i+1}" for i in range(N)]
)

print("\n=== TRANSITION MATRIX ===")
print(tabulate(trans_df, headers="keys", tablefmt="pipe", floatfmt=".4f"))


9. PAGERANK ITERATION

Perhitungan skor LexRank dilakukan menggunakan algoritma PageRank dengan damping factor sebesar 0,85. Proses iterasi dilakukan hingga skor kalimat menunjukkan kestabilan, di mana setiap kalimat diperlakukan sebagai simpul dalam graf dan bobot transisi antar simpul diperoleh dari matriks kemiripan yang telah dinormalisasi.

In [ ]:
d = 0.85
pr = np.ones(N) / N
threshold_error = 1e-6  # 0.000001 (Standar Konvergensi)
max_iter = 100          # Jaga-jaga biar gak loop selamanya

history = []

print(f"\nMulai Iterasi Power Method (Threshold: {threshold_error})...")

for it in range(max_iter):
    prev_pr = pr.copy()

    # Rumus PageRank (LexRank)
    pr = (1 - d)/N + d * row_normalized.T.dot(prev_pr)

    # Simpan history buat tabel
    history.append(pr.copy())

    # Cek Konvergensi (Selisih Error)
    diff = np.sum(np.abs(pr - prev_pr))

    if diff < threshold_error:
        print(f"Konvergen pada iterasi ke-{it+1}")
        break

# Tampilkan Tabel History (Cukup 10 iterasi pertama atau 5 awal + 5 akhir)
if len(history) > 15:
    display_history = history[:5] + history[-5:]
    index_labels = [f"Iterasi {i+1}" for i in range(5)] + \
                   [f"Iterasi {len(history)-4+i}" for i in range(5)]
else:
    display_history = history
    index_labels = [f"Iterasi {i+1}" for i in range(len(history))]

pr_df = pd.DataFrame(display_history, columns=[f"K{i+1}" for i in range(N)])
pr_df.index = index_labels

print("\n=== ITERASI PAGERANK (POWER METHOD) ===")
print(tabulate(pr_df, headers="keys", tablefmt="pipe", floatfmt=".6f"))

10. HASIL RINGKASAN LEXRANK

In [ ]:
scores = list(enumerate(pr))
scores.sort(key=lambda x: x[1], reverse=True)

def show_top_lexrank(top_n, scores, sentences):
    top_scores = scores[:top_n]

    df = pd.DataFrame([
        {
            "Rank": i + 1,
            "Index": idx,
            "Score": score,
            "Kalimat": sentences[idx]
        }
        for i, (idx, score) in enumerate(top_scores)
    ])

    print(f"\n=== HASIL AKHIR LEXRANK (TOP {top_n}) ===")
    print(tabulate(df, headers="keys", tablefmt="pipe", floatfmt=".6f"))

    return df

In [ ]:
lexrank_top_40 = show_top_lexrank(40, scores, sentences)

In [ ]:
lexrank_top_70 = show_top_lexrank(70, scores, sentences)

In [ ]:
def save_summary_txt(summary_sentences, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for sent in summary_sentences:
            f.write(sent.strip() + "\n")

# Simpan hasil LexRank
save_summary_txt(lexrank_top_40, "lexrank_top_40.txt")
save_summary_txt(lexrank_top_70, "lexrank_top_70.txt")

print("Ringkasan LexRank berhasil disimpan.")

12. Evaluasi

In [ ]:
# Ambil Top-N dari LexRank
lexrank_top_40 = show_top_lexrank(40, scores, sentences)
lexrank_top_70 = show_top_lexrank(70, scores, sentences)

system_40 = lexrank_top_40["Kalimat"].tolist()
system_70 = lexrank_top_70["Kalimat"].tolist()

print("System Top-40 (original):", len(system_40))
print("System Top-70 (original):", len(system_70))

In [ ]:
# Fungsi Load Gold Summary
def load_sentences(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


gold_40 = load_sentences("GOL40_PedomanAkademik.txt")
gold_70 = load_sentences("GOL70_PedomanAkademik.txt")

print("Gold Top-40:", len(gold_40))
print("Gold Top-70:", len(gold_70))

In [ ]:
# Normalisasi Kalimat
import re

def normalize_sentence(s):
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)          # hapus spasi ganda
    s = re.sub(r'[^\w\s]', '', s)      # hapus tanda baca
    return s

In [ ]:
# Evaluasi Sentence-Level

def evaluate_sentence_level(system_summary, gold_summary):

    # simpan jumlah awal sesuai Top-N
    original_system_count = len(system_summary)

    # normalisasi dulu supaya perbandingan adil
    system_norm = [normalize_sentence(s) for s in system_summary]
    gold_norm   = [normalize_sentence(s) for s in gold_summary]

    # ubah ke set biar gak ada kalimat dobel
    system_set = set(system_norm)
    gold_set   = set(gold_norm)

    # cari irisan (kalimat yang sama di sistem dan gold)
    intersection = system_set & gold_set

    # Hitung metrik
    precision = len(intersection) / len(system_set) if system_set else 0
    recall    = len(intersection) / len(gold_set) if gold_set else 0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

    return {
        "Original System": original_system_count,
        "Unique System": len(system_set),
        "Gold": len(gold_set),
        "Matched": len(intersection),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4)
    }

In [ ]:
result_40 = evaluate_sentence_level(system_40, gold_40)
result_70 = evaluate_sentence_level(system_70, gold_70)

In [ ]:
import pandas as pd
from tabulate import tabulate

eval_df = pd.DataFrame([
    {"Top-N": 40, **result_40},
    {"Top-N": 70, **result_70}
])

print("\n=== TABEL HASIL EVALUASI LEXRANK ===")
print(tabulate(
    eval_df[["Top-N", "Original System", "Unique System", "Gold",
             "Matched", "Precision", "Recall", "F1-Score"]],
    headers="keys",
    tablefmt="pipe",
    showindex=False
))

In [ ]:
import re
# 1. Normalisasi kalimat
def normalize_sentence(s):
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)          # merapikan spasi berlebih
    s = re.sub(r'[^\w\s]', '', s)      # menghapus tanda baca
    return s

# 2. Menyiapkan data
system_dict = {normalize_sentence(s): s for s in system_70}
gold_dict   = {normalize_sentence(s): s for s in gold_70}

system_keys = set(system_dict.keys())
gold_keys   = set(gold_dict.keys())

# 3. Mengelompokkan hasil
# Kalimat yang cocok antara sistem dan gold
true_positives  = system_keys & gold_keys

# Kalimat penting menurut gold tetapi tidak dipilih sistem
false_negatives = gold_keys - system_keys

# Kalimat yang dipilih sistem tetapi tidak ada di gold
false_positives = system_keys - gold_keys

# 4. Fungsi untuk menampilkan contoh

def print_samples(title, key_set, source_dict, limit=5):
    print(f"\n--- {title} (Total: {len(key_set)}) ---")
    sorted_keys = sorted(list(key_set))  # diurutkan agar rapi
    for i, key in enumerate(sorted_keys[:limit]):
        print(f"{i+1}. {source_dict[key]}")

# 5. Menampilkan hasil analisis

print("=== HASIL ===")

# A. Kasus berhasil
print_samples(
    "A. TRUE POSITIVE (Kalimat sistem yang sesuai dengan gold)",
    true_positives,
    system_dict,
    limit=5
)

# B. Kasus terlewat
print_samples(
    "B. FALSE NEGATIVE (Kalimat gold yang tidak dipilih sistem)",
    false_negatives,
    gold_dict,
    limit=5
)

# C. Kasus salah pilih
print_samples(
    "C. FALSE POSITIVE (Kalimat dipilih sistem tetapi tidak ada di gold)",
    false_positives,
    system_dict,
    limit=5
)


11. GRAF

Visualisasi graf digunakan untuk memberikan gambaran hubungan antar kalimat dengan skor LexRank tertinggi. Graf ini tidak digunakan dalam proses perhitungan LexRank, melainkan hanya sebagai ilustrasi struktur keterkaitan kalimat berdasarkan nilai kemiripan IDF-Modified Cosine


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

k = 40
top_k_idx = np.argsort(pr)[-k:]

G = nx.Graph()

# tambahkan node + simpan skor sebagai atribut
for i in top_k_idx:
    G.add_node(i, score=pr[i])

sim_thresh = 0.15
for a in top_k_idx:
    for b in top_k_idx:
        if a < b:
            w = sim_matrix[a][b]
            if w > sim_thresh:
                G.add_edge(a, b, weight=w)

plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, seed=42)

# ambil skor untuk visualisasi
scores = np.array([G.nodes[n]['score'] for n in G.nodes()])

# ukuran node berdasarkan skor (dikali biar kelihatan beda)
node_sizes = 4000 * scores

# warna node berdasarkan skor
nodes = nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color=scores,
    cmap=plt.cm.viridis
)

# ambil bobot edge
weights = [G[u][v]['weight'] * 3 for u, v in G.edges()]

nx.draw_networkx_edges(
    G,
    pos,
    width=weights,
    alpha=0.6
)

nx.draw_networkx_labels(G, pos, font_size=6)

plt.colorbar(nodes, label="Skor LexRank")
plt.title(f"Visualisasi Graph LexRank (Top {k} Kalimat)")
plt.axis('off')
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# ===============================
# Ambil Top-70 berdasarkan skor LexRank
# ===============================
k = 70
top_k_idx = np.argsort(pr)[-k:]

G = nx.Graph()

# Tambahkan node + simpan skor sebagai atribut
for i in top_k_idx:
    G.add_node(i, score=pr[i])

# ===============================
# Tambahkan edge (threshold 0.15)
# ===============================
sim_thresh = 0.15

for a in top_k_idx:
    for b in top_k_idx:
        if a < b:
            w = sim_matrix[a][b]
            if w > sim_thresh:
                G.add_edge(a, b, weight=w)

# ===============================
# Visualisasi
# ===============================
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, seed=42)

# Ambil skor untuk ukuran & warna
scores = np.array([G.nodes[n]['score'] for n in G.nodes()])

# Ukuran node proporsional skor
node_sizes = 5000 * scores

# Warna node berdasarkan skor
nodes = nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color=scores,
    cmap=plt.cm.plasma
)

# Ketebalan edge berdasarkan bobot similarity
weights = [G[u][v]['weight'] * 2 for u, v in G.edges()]

nx.draw_networkx_edges(
    G,
    pos,
    width=weights,
    alpha=0.5
)

# Label kecil biar tidak terlalu ramai
nx.draw_networkx_labels(G, pos, font_size=6)

plt.colorbar(nodes, label="Skor LexRank")
plt.title(f"Visualisasi Graph LexRank (Top {k} Kalimat)")
plt.axis('off')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. DATA SIMULASI (Hanya untuk ilustrasi konsep Trade-off)
# Rentang jumlah kalimat (dari 30 sampai 80 kalimat)
x = np.linspace(30, 80, 100)

# Kurva Kelengkapan Informasi (Recall/Completeness)
# Semakin banyak kalimat, informasi makin lengkap (naik melengkung)
y_kelengkapan = 1 - 0.7 * np.exp(-0.06 * (x - 30))

# Kurva Kepadatan/Efisiensi (Conciseness/Precision)
# Semakin banyak kalimat, redundansi naik, jadi efisiensi turun
y_kepadatan = 0.9 * np.exp(-0.03 * (x - 30))

# 2. MEMBUAT PLOT
plt.figure(figsize=(10, 6)) # Ukuran gambar

# Plot garis
plt.plot(x, y_kelengkapan, label='Kelengkapan Informasi (Recall)', color='blue', linewidth=2.5)
plt.plot(x, y_kepadatan, label='Kepadatan & Efisiensi', color='orange', linewidth=2.5, linestyle='--')

# 3. MENANDAI TITIK TOP-40 dan TOP-70
# Titik Top-40 (Lebih padat, tapi info kurang lengkap)
plt.axvline(x=40, color='grey', linestyle=':', alpha=0.6)
plt.text(40, 0.45, ' Top-40\n (Padat, Info Parsial)', verticalalignment='center', fontsize=10)

# Titik Top-70 (Info lengkap, tapi redundansi tinggi/kurang padat)
plt.axvline(x=70, color='grey', linestyle=':', alpha=0.6)
plt.text(70, 0.45, ' Top-70\n (Lengkap, Redundan)', verticalalignment='center', fontsize=10)

# 4. PERCANTIK GRAFIK
plt.title('Trade-off Antara Kelengkapan dan Kepadatan Ringkasan', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Ukuran Ringkasan (Jumlah Kalimat)', fontsize=12)
plt.ylabel('Tingkat Kinerja (Ilustrasi)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.ylim(0, 1.1) # Batas sumbu Y

# Simpan gambar (Opsional, bisa di-download)
plt.savefig('Grafik_Tradeoff_LexRank.png', dpi=300)

# Tampilkan
plt.show()